# Лабораторная работа №3 по ТМО
## [Ваше Имя Фамилия]
## Группа [Ваша группа]

#### Цель работы. Изучение способов подготовки выборки и подбора гиперпараметров на примере метода ближайших соседей.

## 1. Импорт библиотек и загрузка датасета

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV,
    StratifiedKFold, RepeatedStratifiedKFold
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Загрузка датасета
wine = load_wine()
X = wine.data
y = wine.target
feature_names = wine.feature_names
target_names = wine.target_names

print('Датасет: Wine Recognition Dataset')
print(f'Задача: классификация сортов вина')
print(f'Количество образцов: {X.shape[0]}')
print(f'Количество признаков: {X.shape[1]}')
print(f'Классы (сорта вина): {list(target_names)}')

#### В качестве набора данных выбран датасет **Wine Recognition** — результаты химического анализа 178 образцов вин трёх сортов из Италии. Каждый образец описывается 13 числовыми химическими характеристиками. Задача — классификация: определить сорт вина по химическому составу.

## 2. Проверка наличия пропусков и предобработка данных

In [ ]:
# Проверка пропусков
print('=== Проверка данных ===')
print(f'Пропуски в признаках: {np.isnan(X).sum()}')
print(f'Пропуски в целевой переменной: {np.isnan(y).sum()}')

# Распределение классов
print('\n=== Распределение классов ===')
unique, counts = np.unique(y, return_counts=True)
for cls, cnt in zip(target_names, counts):
    print(f'  {cls}: {cnt} образцов')

# Описательная статистика
df = pd.DataFrame(X, columns=feature_names)
print('\n=== Описательная статистика (первые 4 признака) ===')
print(df.iloc[:, :4].describe().round(2))

In [ ]:
# Нормализация признаков — ОБЯЗАТЕЛЬНА для KNN
# KNN вычисляет расстояния: признаки с разными масштабами будут влиять неравномерно
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('=== Нормализация признаков (StandardScaler) ===')
print(f'До:    среднее alcohol = {X[:, 0].mean():.2f}, std = {X[:, 0].std():.2f}')
print(f'После: среднее alcohol = {X_scaled[:, 0].mean():.4f}, std = {X_scaled[:, 0].std():.4f}')
print('\nПропусков нет, категориальных признаков нет — дополнительная обработка не требуется.')

#### Пропусков в данных нет, все признаки числовые. Однако для KNN **обязательна нормализация**: алгоритм вычисляет расстояния между объектами, поэтому признаки с большим диапазоном значений будут неоправданно доминировать над остальными.

## 3. Разделение выборки на обучающую и тестовую

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42, stratify=y
)

print('=== Разделение выборки ===')
print(f'Обучающая выборка: {X_train.shape[0]} образцов ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Тестовая выборка:  {X_test.shape[0]} образцов ({X_test.shape[0]/len(X)*100:.0f}%)')
print('\nПараметр stratify=y обеспечивает пропорциональное представление классов.')

print('\n=== Распределение классов после разбиения ===')
for i, cls in enumerate(target_names):
    tr = np.sum(y_train == i)
    te = np.sum(y_test == i)
    print(f'  {cls}: обучение={tr}, тест={te}')

## 4. Обучение модели KNN для произвольно заданного гиперпараметра K

In [ ]:
K_initial = 5  # произвольное начальное значение

knn_initial = KNeighborsClassifier(n_neighbors=K_initial)
knn_initial.fit(X_train, y_train)

y_pred_initial = knn_initial.predict(X_test)

acc_initial = accuracy_score(y_test, y_pred_initial)
f1_initial = f1_score(y_test, y_pred_initial, average='weighted')

print(f'=== Модель с K = {K_initial} ===')
print(f'Accuracy:            {acc_initial:.4f}')
print(f'F1-score (weighted): {f1_initial:.4f}')
print('\nПодробный отчёт:')
print(classification_report(y_test, y_pred_initial, target_names=target_names))

In [ ]:
# Матрица ошибок для начальной модели
cm = confusion_matrix(y_test, y_pred_initial)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.title(f'Матрица ошибок — начальная модель (K={K_initial})', fontsize=13)
plt.ylabel('Истинный класс', fontsize=11)
plt.xlabel('Предсказанный класс', fontsize=11)
plt.tight_layout()
plt.show()

#### При K=5 модель показывает хорошее качество. Попробуем найти оптимальный K с помощью кросс-валидации.

## 5. Подбор гиперпараметра K: GridSearchCV и RandomizedSearchCV

### Стратегия 1. GridSearchCV + StratifiedKFold

In [ ]:
k_range = list(range(1, 31))  # K от 1 до 30
param_grid = {'n_neighbors': k_range}

# StratifiedKFold сохраняет пропорции классов в каждом фолде
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=skf,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

best_k_grid = grid_search.best_params_['n_neighbors']
best_cv_score_grid = grid_search.best_score_

print('=== GridSearchCV + StratifiedKFold (5 фолдов) ===')
print(f'Лучший K:           {best_k_grid}')
print(f'Лучший CV Accuracy: {best_cv_score_grid:.4f}')

best_knn_grid = grid_search.best_estimator_
y_pred_grid = best_knn_grid.predict(X_test)
acc_grid = accuracy_score(y_test, y_pred_grid)
f1_grid = f1_score(y_test, y_pred_grid, average='weighted')

print(f'\nКачество на тестовой выборке:')
print(f'  Accuracy: {acc_grid:.4f}')
print(f'  F1-score: {f1_grid:.4f}')

In [ ]:
# График CV Accuracy по K
cv_results = grid_search.cv_results_
k_values = cv_results['param_n_neighbors'].data.astype(int)
mean_scores = cv_results['mean_test_score']
std_scores = cv_results['std_test_score']

plt.figure(figsize=(10, 5))
plt.plot(k_values, mean_scores, 'o-', color='steelblue', linewidth=2, markersize=5, label='CV Accuracy')
plt.fill_between(k_values, mean_scores - std_scores, mean_scores + std_scores,
                 alpha=0.2, color='steelblue', label='+-std')
plt.axvline(x=best_k_grid, color='red', linestyle='--', linewidth=2, label=f'Лучший K={best_k_grid}')
plt.xlabel('Значение K', fontsize=12)
plt.ylabel('Accuracy (кросс-валидация)', fontsize=12)
plt.title('GridSearchCV: зависимость точности от K (StratifiedKFold, 5 фолдов)', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Стратегия 2. RandomizedSearchCV + RepeatedStratifiedKFold

In [ ]:
# RepeatedStratifiedKFold: 5 фолдов, повторяется 3 раза — итого 15 оценок на комбинацию
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

# Расширенное пространство: K + метрика расстояния
param_dist = {
    'n_neighbors': list(range(1, 31)),
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

random_search = RandomizedSearchCV(
    KNeighborsClassifier(),
    param_dist,
    cv=rskf,
    scoring='accuracy',
    n_iter=30,
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train, y_train)

best_k_random = random_search.best_params_['n_neighbors']
best_metric_random = random_search.best_params_['metric']
best_cv_score_random = random_search.best_score_

print('=== RandomizedSearchCV + RepeatedStratifiedKFold (5x3) ===')
print(f'Лучший K:                   {best_k_random}')
print(f'Лучшая метрика расстояния:  {best_metric_random}')
print(f'Лучший CV Accuracy:         {best_cv_score_random:.4f}')

best_knn_random = random_search.best_estimator_
y_pred_random = best_knn_random.predict(X_test)
acc_random = accuracy_score(y_test, y_pred_random)
f1_random = f1_score(y_test, y_pred_random, average='weighted')

print(f'\nКачество на тестовой выборке:')
print(f'  Accuracy: {acc_random:.4f}')
print(f'  F1-score: {f1_random:.4f}')
print('\nПодробный отчёт:')
print(classification_report(y_test, y_pred_random, target_names=target_names))

In [ ]:
# Матрица ошибок для лучшей модели (RandomizedSearchCV)
cm_random = confusion_matrix(y_test, y_pred_random)

plt.figure(figsize=(7, 5))
sns.heatmap(cm_random, annot=True, fmt='d', cmap='Greens',
            xticklabels=target_names, yticklabels=target_names)
plt.title(f'Матрица ошибок — лучшая модель (K={best_k_random}, {best_metric_random})', fontsize=13)
plt.ylabel('Истинный класс', fontsize=11)
plt.xlabel('Предсказанный класс', fontsize=11)
plt.tight_layout()
plt.show()

## 6. Сравнение метрик качества моделей

In [ ]:
# Сводная таблица
df_results = pd.DataFrame({
    'Модель': [
        f'Исходная (K={K_initial})',
        f'GridSearchCV (K={best_k_grid})',
        f'RandomizedSearchCV (K={best_k_random}, {best_metric_random})'
    ],
    'Accuracy': [acc_initial, acc_grid, acc_random],
    'F1-score': [f1_initial, f1_grid, f1_random]
})

print('=== Сравнение моделей ===')
print(df_results.to_string(index=False))

In [ ]:
model_names = [
    f'Исходная\n(K={K_initial})',
    f'GridSearchCV\n(K={best_k_grid})',
    f'RandomizedSearchCV\n(K={best_k_random})'
]
colors = ['steelblue', 'darkgreen', 'darkorange']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

acc_values = [acc_initial, acc_grid, acc_random]
bars1 = axes[0].bar(model_names, acc_values, color=colors, edgecolor='black', linewidth=1.2)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Сравнение Accuracy', fontsize=13)
axes[0].set_ylim(0, 1.15)
axes[0].grid(True, alpha=0.3, axis='y')
for bar, v in zip(bars1, acc_values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.02, f'{v:.4f}', ha='center', fontsize=11)

f1_values = [f1_initial, f1_grid, f1_random]
bars2 = axes[1].bar(model_names, f1_values, color=colors, edgecolor='black', linewidth=1.2)
axes[1].set_ylabel('F1-score (weighted)', fontsize=12)
axes[1].set_title('Сравнение F1-score', fontsize=13)
axes[1].set_ylim(0, 1.15)
axes[1].grid(True, alpha=0.3, axis='y')
for bar, v in zip(bars2, f1_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.02, f'{v:.4f}', ha='center', fontsize=11)

plt.suptitle('Сравнение качества моделей KNN', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Вывод

#### В ходе лабораторной работы метод K ближайших соседей (KNN) применён к задаче классификации сортов вина. Нормализация данных обязательна для KNN, так как алгоритм основан на расстояниях.

#### Подбор K с помощью GridSearchCV (StratifiedKFold, 5 фолдов) и RandomizedSearchCV (RepeatedStratifiedKFold, 5×3) позволил найти оптимальное значение K. RandomizedSearchCV дополнительно подобрал лучшую метрику расстояния.

#### Оптимизированные модели показали более высокое качество по сравнению с исходной моделью при произвольном K=5, что подтверждает важность систематического подбора гиперпараметров с кросс-валидацией.